# MAPOCA clean pipeline - Ice Hockey (PettingZoo)

Pipeline unique et nettoyee:
1. Setup
2. Environment
3. Actor/Critic
4. Collecte
5. Update
6. Entrainement
7. Evaluation + GIF


In [ ]:
# Si besoin dans Colab:
# !pip install pettingzoo[atari] supersuit multi-agent-ale-py AutoROM
# !AutoROM --accept-license

import random
import numpy as np
import matplotlib.pyplot as plt
import PIL.Image

import torch
import torch.nn.functional as F
import torch.optim as optim

import supersuit as ss
from pettingzoo.atari import ice_hockey_v2
from IPython.display import Image, display

from MAPOCA.actor import MultiAgentActors
from MAPOCA.centralized_critic import Centralized_critic
from MAPOCA.multi_agent_buffer import MultiAgentBuffer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


ModuleNotFoundError: No module named 'supersuit'

In [ ]:
def make_env(render_mode=None):
    env = ice_hockey_v2.env(render_mode=render_mode)
    env = ss.color_reduction_v0(env, mode="R")
    env = ss.resize_v1(env, x_size=84, y_size=84)
    env = ss.frame_stack_v1(env, 4)
    env = ss.dtype_v0(env, np.float32)
    return env

env_check = make_env()
env_check.reset(seed=SEED)
obs, reward, term, trunc, info = env_check.last()
print("Observation shape:", obs.shape)
env_check.close()


In [ ]:
def _safe_ram_xy(ram, puck_idx_x, puck_idx_y, player_idx_x, player_idx_y):
    # Conversion en float pour eviter les overflows uint8 dans les soustractions
    x_puck = float(ram[puck_idx_x])
    y_puck = float(ram[puck_idx_y])
    x_player = float(ram[player_idx_x])
    y_player = float(ram[player_idx_y])
    return x_puck, y_puck, x_player, y_player

def shaped_reward(agent, env_reward, ram):
    if agent == "first_0":
        x_puck, y_puck, x_j, y_j = _safe_ram_xy(ram, 54, 55, 56, 57)
        direction = 0.10 if x_puck > 80.0 else -0.02
        defense_penalty = -0.08 if x_j < 20.0 else 0.0
    else:
        x_puck, y_puck, x_j, y_j = _safe_ram_xy(ram, 54, 55, 58, 59)
        direction = 0.10 if x_puck < 80.0 else -0.02
        defense_penalty = -0.08 if x_j > 140.0 else 0.0

    dist = np.sqrt((x_j - x_puck) ** 2 + (y_j - y_puck) ** 2)
    distance_reward = 1.0 / (dist + 1.0)

    event_reward = float(env_reward) * 50.0
    time_penalty = -0.005
    total = event_reward + 2.0 * distance_reward + direction + defense_penalty + time_penalty
    return float(total)


In [ ]:
def collect_trajectories(env, actor, buffer, max_steps=256):
    env.reset(seed=SEED)
    actor.eval()

    temp_obs = {}
    temp_actions = {}
    temp_logprobs = {}
    temp_rewards = {}

    step_count = 0

    for agent in env.agent_iter():
        obs, env_reward, terminated, truncated, _ = env.last()
        done = bool(terminated or truncated)

        ram = env.unwrapped.ale.getRAM()
        temp_rewards[agent] = shaped_reward(agent, env_reward, ram)

        if done:
            action = None
            logp = 0.0
        else:
            obs_chw = obs.transpose(2, 0, 1)
            obs_tensor = torch.from_numpy(obs_chw).float().unsqueeze(0).to(device)
            with torch.no_grad():
                logits = actor(obs_tensor)
                dist = torch.distributions.Categorical(logits=logits)
                a = dist.sample()
                action = int(a.item())
                logp = float(dist.log_prob(a).item())

        env.step(action)
        temp_obs[agent] = obs

        if action is not None:
            temp_actions[agent] = action
            temp_logprobs[agent] = logp

        # Un pas de temps global = les 2 agents ont agi
        if len(temp_actions) == 2:
            if "first_0" in temp_obs and "second_0" in temp_obs:
                buffer.add(
                    observation=[temp_obs["first_0"], temp_obs["second_0"]],
                    action=[temp_actions["first_0"], temp_actions["second_0"]],
                    logprob=[temp_logprobs["first_0"], temp_logprobs["second_0"]],
                    reward=[temp_rewards.get("first_0", 0.0), temp_rewards.get("second_0", 0.0)],
                    done=done,
                )
                step_count += 1

            temp_actions = {}
            temp_logprobs = {}

        if done or step_count >= max_steps:
            break

    return step_count


In [ ]:
def compute_returns_lambda(rewards, values, next_value, dones, gamma=0.99, lmbda=0.95):
    returns = torch.zeros_like(rewards)
    gae = torch.tensor(0.0, device=rewards.device)

    for t in reversed(range(len(rewards))):
        if t == len(rewards) - 1:
            next_non_terminal = 1.0 - dones[t]
            next_val = next_value
        else:
            next_non_terminal = 1.0 - dones[t + 1]
            next_val = values[t + 1]

        delta = rewards[t] + gamma * next_val * next_non_terminal - values[t]
        gae = delta + gamma * lmbda * next_non_terminal * gae
        returns[t] = gae + values[t]

    return returns

def update_mapoca(actor, critic, buffer, optimizer_actor, optimizer_critic, gamma=0.99, lmbda=0.95, entropy_coef=0.01):
    obs_raw_1 = torch.stack([torch.from_numpy(o[0]) for o in buffer.observations]).float()
    obs_raw_2 = torch.stack([torch.from_numpy(o[1]) for o in buffer.observations]).float()
    obs_ag1 = obs_raw_1.permute(0, 3, 1, 2).to(device)
    obs_ag2 = obs_raw_2.permute(0, 3, 1, 2).to(device)

    rewards = torch.tensor(buffer.rewards, dtype=torch.float32, device=device)
    dones = torch.tensor(buffer.dones, dtype=torch.float32, device=device)
    actions = torch.tensor(buffer.actions, dtype=torch.long, device=device)

    with torch.no_grad():
        values = critic([obs_ag1, obs_ag2])
        next_value = torch.zeros(2, dtype=torch.float32, device=device)
        returns_0 = compute_returns_lambda(rewards[:, 0], values[:, 0], next_value[0], dones, gamma, lmbda)
        returns_1 = compute_returns_lambda(rewards[:, 1], values[:, 1], next_value[1], dones, gamma, lmbda)
        returns = torch.stack([returns_0, returns_1], dim=1)
        advantages = returns - values
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

    current_values = critic([obs_ag1, obs_ag2])
    critic_loss = F.mse_loss(current_values, returns)
    optimizer_critic.zero_grad()
    critic_loss.backward()
    torch.nn.utils.clip_grad_norm_(critic.parameters(), 0.5)
    optimizer_critic.step()

    logits1 = actor(obs_ag1)
    dist1 = torch.distributions.Categorical(logits=logits1)
    logp1 = dist1.log_prob(actions[:, 0])

    logits2 = actor(obs_ag2)
    dist2 = torch.distributions.Categorical(logits=logits2)
    logp2 = dist2.log_prob(actions[:, 1])

    actor_loss_pg = -(logp1 * advantages[:, 0]).mean() - (logp2 * advantages[:, 1]).mean()
    entropy = dist1.entropy().mean() + dist2.entropy().mean()
    actor_loss = actor_loss_pg - entropy_coef * entropy

    optimizer_actor.zero_grad()
    actor_loss.backward()
    torch.nn.utils.clip_grad_norm_(actor.parameters(), 0.5)
    optimizer_actor.step()

    avg_batch_reward = float(rewards.mean().item())
    return float(actor_loss_pg.item()), float(critic_loss.item()), avg_batch_reward


In [ ]:
# Hyperparametres
LR_ACTOR = 1e-4
LR_CRITIC = 3e-4
GAMMA = 0.99
LMBDA = 0.95
BATCH_STEPS = 256
TOTAL_UPDATES = 400
ENTROPY_COEF = 0.01
LOG_EVERY = 10

train_env = make_env(render_mode=None)
actor = MultiAgentActors(action_dim=18).to(device)
critic = Centralized_critic(num_agents=2).to(device)
optimizer_actor = optim.Adam(actor.parameters(), lr=LR_ACTOR)
optimizer_critic = optim.Adam(critic.parameters(), lr=LR_CRITIC)
buffer = MultiAgentBuffer()

actor_losses = []
critic_losses = []
batch_rewards = []

for update in range(1, TOTAL_UPDATES + 1):
    buffer.clear()
    steps = collect_trajectories(train_env, actor, buffer, max_steps=BATCH_STEPS)

    if len(buffer.observations) < 16:
        continue

    a_loss, c_loss, avg_r = update_mapoca(
        actor,
        critic,
        buffer,
        optimizer_actor,
        optimizer_critic,
        gamma=GAMMA,
        lmbda=LMBDA,
        entropy_coef=ENTROPY_COEF,
    )

    actor_losses.append(a_loss)
    critic_losses.append(c_loss)
    batch_rewards.append(avg_r)

    if update % LOG_EVERY == 0:
        print(f"Update {update:4d} | steps {steps:3d} | actor_loss {a_loss:8.4f} | critic_loss {c_loss:8.4f} | avg_batch_reward {avg_r:8.4f}")

train_env.close()


In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.plot(actor_losses)
plt.title("Actor loss")

plt.subplot(1, 3, 2)
plt.plot(critic_losses)
plt.yscale("log")
plt.title("Critic loss (log)")

plt.subplot(1, 3, 3)
plt.plot(batch_rewards)
plt.title("Average batch reward")

plt.tight_layout()
plt.show()


In [ ]:
def record_video(env, actor, filename="mapoca_clean_eval.gif", max_agent_steps=2000, frame_every=2):
    actor.eval()
    env.reset(seed=SEED)
    frames = []

    for i, agent in enumerate(env.agent_iter()):
        obs, reward, terminated, truncated, _ = env.last()

        if i % frame_every == 0:
            frame = env.render()
            if frame is not None:
                frames.append(PIL.Image.fromarray(frame))

        if terminated or truncated:
            action = None
        else:
            obs_chw = obs.transpose(2, 0, 1)
            obs_tensor = torch.from_numpy(obs_chw).float().unsqueeze(0).to(device)
            with torch.no_grad():
                action = int(actor.get_action(obs_tensor, deterministic=True).item())

        env.step(action)

        if terminated or truncated or i >= max_agent_steps:
            break

    if len(frames) == 0:
        raise RuntimeError("Aucune frame capturee.")

    frames[0].save(filename, save_all=True, append_images=frames[1:], duration=40, loop=0)
    return filename, len(frames)

eval_env = make_env(render_mode="rgb_array")
gif_path, n_frames = record_video(eval_env, actor, filename="mapoca_clean_eval.gif")
eval_env.close()

print("GIF saved:", gif_path, "| frames:", n_frames)
display(Image(filename=gif_path))
